# Order_payments Feature Engineering

## Objective

This notebook creates business-oriented features from the cleaned `order_payments` dataset.

In [1]:
# Loading data
import pandas as pd
import numpy as np

order_payments = pd.read_csv(
    "../../03_Python_ETL/Output/order_payments_clean.csv"
)

In [2]:
order_payments.shape

(103886, 5)

In [3]:
order_payments.describe(include="all")

,order_id,payment_sequential,payment_type,payment_installments,payment_value
count,103886,103886.000000,103886,103886.000000,103886.000000
unique,99440,NaN,5,NaN,NaN
top,fa65dad1b0e818e3ccc5cb0e39231352,NaN,credit_card,NaN,NaN
freq,29,NaN,76795,NaN,NaN
mean,NaN,1.092679,NaN,2.853349,154.100380
std,NaN,0.706584,NaN,2.687051,217.494064
min,NaN,1.000000,NaN,0.000000,0.000000
25%,NaN,1.000000,NaN,1.000000,56.790000
50%,NaN,1.000000,NaN,1.000000,100.000000
75%,NaN,1.000000,NaN,4.000000,171.837500


### Feature 1: payment_total_value_per_order

**Business Purpose:**
Calculates the total amount paid for an order by summing all payment transactions associated with the same order_id. This helps measure actual order revenue and identify orders split across multiple payment transactions.


In [4]:
# Pyhthon Implementation
order_payments["payment_total_value_per_order"] = (
    order_payments.groupby("order_id")["payment_value"]
    .transform("sum")
)

In [5]:
# Preview
order_payments[
    ["order_id", "payment_value", "payment_total_value_per_order"]
].head()

,order_id,payment_value,payment_total_value_per_order
0,b81ef226f3fe1789b1e8b2acac839d17,99.33,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,24.39,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,65.71,65.71
3,ba78997921bbcdc1373bb41e913ab953,107.78,107.78
4,42fdf880ba16b47b59251dd489d4441a,128.45,128.45


In [6]:
# Summary statistics
order_payments["payment_total_value_per_order"].describe()

count    103886.000000
mean        161.155179
std         222.384248
min           0.000000
25%          62.010000
50%         105.200000
75%         176.780000
max       13664.080000
Name: payment_total_value_per_order, dtype: float64

In [7]:
order_payments["payment_total_value_per_order"].isnull().sum()

0

### Feature 2: payment_method_count

**Business Purpose:**
Calculates the number of distinct payment methods used for each order. This helps identify whether customers completed their purchase using a single payment method or split the payment across multiple methods.

In [8]:
# Pyhthon Implementation
order_payments["payment_method_count"] = (
    order_payments.groupby("order_id")["payment_type"]
    .transform("nunique")
)

In [9]:
# Validation
order_payments[
    ["order_id", "payment_type", "payment_method_count"]
].drop_duplicates().sort_values(by = 'payment_method_count',ascending= False).head(10)

,order_id,payment_type,payment_method_count
70662,b8ccf087dea4fc431e71277cf66ed793,credit_card,2
87281,a3b45c3744c4a07892d51bc29f6b1281,credit_card,2
7031,05e6d57ea1f64b4e9a0b37c669515cfc,voucher,2
10022,0b446830af3811ef0d7c2e0db238af50,credit_card,2
32902,6b1506319e933481532aba2b6de29d12,credit_card,2
87277,3e1298fc81396a1e3ad660c0cd0fab65,credit_card,2
87279,922587c6bf637f545a82eb3f9136ae30,credit_card,2
61751,b37ae92c72d3335c241ea5be05dbc077,credit_card,2
3628,898ded7ecb1e4d92997856bd8b7a8aa1,voucher,2
91793,a1ad76f125118edcc2ba7035e9a59c7c,voucher,2


In [10]:
order_payments["payment_method_count"].describe()

count    103886.000000
mean          1.051855
std           0.221735
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           2.000000
Name: payment_method_count, dtype: float64

In [11]:
order_payments["payment_method_count"].isnull().sum()

0

### Feature 3: multiple_payment_methods

**Business Purpose:**
Identifies whether an order was paid using more than one payment method. This helps analyze customer payment behavior and the frequency of split-payment transactions.

In [12]:
order_payments["multiple_payment_methods"] = (
    order_payments["payment_method_count"] > 1
).astype(int)

In [13]:
# Validation
order_payments[
    [
        "order_id",
        "payment_method_count",
        "multiple_payment_methods"]].head(10)

,order_id,payment_method_count,multiple_payment_methods
0,b81ef226f3fe1789b1e8b2acac839d17,1,0
1,a9810da82917af2d9aefd1278f1dcfa0,1,0
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,0
3,ba78997921bbcdc1373bb41e913ab953,1,0
4,42fdf880ba16b47b59251dd489d4441a,1,0
5,298fcdf1f73eb413e4d26d01b25bc1cd,1,0
6,771ee386b001f06208a7419e4fc1bbd7,1,0
7,3d7239c394a212faae122962df514ac7,1,0
8,1f78449c87a54faf9e96e88ba1491fa9,1,0
9,0573b5e23cbd798006520e1d5b4c6714,1,0


In [14]:
order_payments["multiple_payment_methods"].value_counts()

multiple_payment_methods
0    98499
1     5387
Name: count, dtype: int64

In [15]:
order_payments["multiple_payment_methods"].isnull().sum()

0

### Feature 4: max_installments_per_order

**Business Purpose:**
Calculates the highest number of installments used for an order across all its payment transactions. This helps analyze customer financing behavior and identify orders with longer repayment periods.

In [16]:
order_payments["max_installments_per_order"] = (
    order_payments.groupby("order_id")["payment_installments"]
    .transform("max")
)

In [17]:
# Validation
order_payments[
    [
        "order_id",
        "payment_installments",
        "max_installments_per_order"
    ]].sort_values(by="max_installments_per_order",
    ascending=False).head(10)

,order_id,payment_installments,max_installments_per_order
21713,6ae2e8b8fac02522481d2a2f4ca4412c,24,24
39098,2b7dbe9be72b8f9733844c31055c0825,1,24
10791,859f516f2fc3f95772e63c5757ab0d5b,24,24
75038,90f864fe19d11549fa01eb81c4dd87e3,1,24
52846,63dbe0c8e63e5f1b4deec09d4f044a7f,24,24
87593,61450e6c8f56d52e46a198e57df7d731,24,24
66746,fe808fc011ee4ae41f2ed8d1d52b6670,24,24
2970,70b7e94ea46d3e8b5bc12a50186edaf0,24,24
102008,f60ce04ff8060152c83c7c97e246d6a8,24,24
102435,e02d61b42452cc6737650331d8bc8ad7,24,24


In [18]:
order_payments["max_installments_per_order"].describe()

count    103886.000000
mean          2.887194
std           2.701435
min           0.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          24.000000
Name: max_installments_per_order, dtype: float64

In [19]:
order_payments["max_installments_per_order"].isnull().sum()

0

### Feature 5: installment_payment_flag

**Business Purpose:**
Identifies whether an order was paid using installments. This helps analyze customer financing behavior and distinguish between one-time payments and installment-based purchases.

In [20]:
order_payments["installment_payment_flag"] = (
    order_payments["max_installments_per_order"] > 1
).astype(int)

In [21]:
# Validation
order_payments[
    [
        "order_id",
        "max_installments_per_order",
        "installment_payment_flag"
    ]
].sort_values(
    by="max_installments_per_order",
    ascending=False
).head(10)

,order_id,max_installments_per_order,installment_payment_flag
21713,6ae2e8b8fac02522481d2a2f4ca4412c,24,1
39098,2b7dbe9be72b8f9733844c31055c0825,24,1
10791,859f516f2fc3f95772e63c5757ab0d5b,24,1
75038,90f864fe19d11549fa01eb81c4dd87e3,24,1
52846,63dbe0c8e63e5f1b4deec09d4f044a7f,24,1
87593,61450e6c8f56d52e46a198e57df7d731,24,1
66746,fe808fc011ee4ae41f2ed8d1d52b6670,24,1
2970,70b7e94ea46d3e8b5bc12a50186edaf0,24,1
102008,f60ce04ff8060152c83c7c97e246d6a8,24,1
102435,e02d61b42452cc6737650331d8bc8ad7,24,1


In [22]:
order_payments["installment_payment_flag"].value_counts()

installment_payment_flag
1    52293
0    51593
Name: count, dtype: int64

In [23]:
order_payments["installment_payment_flag"].isnull().sum()

0

### Feature 6: average_payment_value_per_transaction

**Business Purpose:**
Calculates the average value of each payment transaction for an order. This helps understand how the total payment amount is distributed across multiple payment transactions and can identify orders split into several smaller payments.

In [24]:
order_payments["average_payment_value_per_transaction"] = (
    order_payments.groupby("order_id")["payment_value"]
    .transform("mean")
)

In [25]:
order_payments[
    [
        "order_id",
        "payment_value",
        "average_payment_value_per_transaction"
    ]
].sort_values(
    by="average_payment_value_per_transaction",
    ascending=False
).head(10)

,order_id,payment_value,average_payment_value_per_transaction
52107,03caa2c082116e1d31e67e9ae3700499,13664.08,13664.08
34370,736e1922ae60d0d6a89247b851902527,7274.88,7274.88
41419,0812eb902a67711a1cb742b3cdaa65ae,6929.31,6929.31
49581,fefacc66af859508bf1a7934eab1e97f,6922.21,6922.21
85539,f5136e38d1a14a4dbd87dff67da82701,6726.66,6726.66
62409,2cc9089445046817a7539d90805e6e5a,6081.54,6081.54
43232,a96610ab360d42a2e5335a3998b4718a,4950.34,4950.34
70320,b4c4b76c642808cbe472a32b86cddc95,4809.44,4809.44
6440,199af31afc78c699f0dbf71fb178d4d4,4764.34,4764.34
67546,8dbc85d1447242f3b127dda390d56e19,4681.78,4681.78


In [26]:
order_payments["average_payment_value_per_transaction"].describe()

count    103886.000000
mean        154.100380
std         216.473375
min           0.000000
25%          57.130000
50%          99.820000
75%         171.510000
max       13664.080000
Name: average_payment_value_per_transaction, dtype: float64

In [27]:
order_payments["average_payment_value_per_transaction"].isnull().sum()

0

In [28]:
order_payments.to_csv('../Output/order_payments_features.csv', index = False)